# Adversarial Search: Solving Tic-Tac-Toe with Heuristic Alpha-Beta Tree Search

## Introduction 

Multiplayer games can be implemented as:
1. Nondeterministic actions: The opponent is seen as part of an environment with nondeterministic actions. Non-determinism is the result of the unknown opponent's moves. 
2. Optimal Decisions: Minimax search (search complete game tree) and alpha-beta pruning.
3. __Heuristic Alpha-Beta Tree Search:__ Cut off tree search and use heuristic to estimate state value. 
4. Monte Carlo Tree search: Simulate playouts to estimate state value. 

Here we will implement search for Tic-Tac-Toe (see [rules](https://en.wikipedia.org/wiki/Tic-tac-toe)). The game is a __zero-sum game__: Win by x results in +1, win by o in -1 and a tie has a value of 0. Max plays x and tries to maximize the outcome while Min plays o and tries to minimize the outcome.   

We will implement
* Heuristic Alpha-Beta Tree Search

The algorithms search the game tree and we could return a conditional plan (or partial plan if cut offs are used), but the implementation here only identifies and returns the optimal next move.

Môi trường Tic-Tac-Toe và các hàm trạng thái được cài đặt trực tiếp trong notebook để notebook có thể chạy độc lập, không phụ thuộc file `tictactoe.py`.

In [17]:
import os, sys, pathlib, numpy as np, pandas as pd, math, time, random
from collections import Counter
# Self-contained implementation: no external tictactoe.py dependency.
np.random.seed(2026); random.seed(2026)

from functools import lru_cache

import matplotlib.pyplot as plt
import os

# Tạo thư mục result tại thư mục đang chạy notebook
RESULT_DIR = "result"
os.makedirs(RESULT_DIR, exist_ok=True)

print(f"Thư mục lưu kết quả: {os.path.abspath(RESULT_DIR)}")

def empty_board(): return [' ']*9
def actions(board): return [i for i,v in enumerate(board) if v==' ']
def result(state,player,action):
    if action not in actions(state): raise ValueError(f"Illegal move {action}")
    ns=list(state); ns[action]=player; return ns
def other(player): return 'o' if player=='x' else 'x'
def check_board(state):
    lines=((0,1,2),(3,4,5),(6,7,8),(0,3,6),(1,4,7),(2,5,8),(0,4,8),(2,4,6))
    for a,b,c in lines:
        if state[a]!=' ' and state[a]==state[b]==state[c]: return state[a]
    return 'd' if ' ' not in state else 'n'
def terminal(state): return check_board(state)!='n'
def utility(state,player='x'):
    outcome=check_board(state)
    if outcome==player: return 1
    if outcome==other(player): return -1
    if outcome=='d': return 0
    return None
def show_board(board):
    print(np.asarray(board).reshape(3,3))


def random_player(board, player='x'):
    return random.choice(actions(board))

def play(player1, player2, N=100):
    """Play N games; alternate who moves first and return aggregate results."""
    wins={'x':0,'o':0,'d':0}
    for game in range(N):
        board=empty_board()
        current='x'
        first,second=(player1,player2) if game%2==0 else (player2,player1)
        while not terminal(board):
            agent=first if current=='x' else second
            move=agent(board,current)
            if move not in actions(board):
                raise ValueError(f"Illegal move {move}")
            board=result(board,current,move)
            current=other(current)
        outcome=check_board(board)
        wins[outcome]+=1
    return wins


Thư mục lưu kết quả: f:\TTNTNC\LAB05_GAME\result


In [18]:
# The environment and helper functions are already defined in the preceding cell.
# Keep this cell as a dependency check rather than re-importing an external module.
assert len(empty_board()) == 9
assert len(actions(empty_board())) == 9
print("Self-contained Tic-Tac-Toe environment is ready.")


Self-contained Tic-Tac-Toe environment is ready.


# Heuristic Alpha-Beta Tree Search

See AIMA page 156ff. 


## Heuristic Evaluation Function

In [19]:
import numpy as np

def eval_fun(state, player = 'x'):
    """heuristic for utility of state. Returns score for a node:
    1. For terminal states it returns the utility. 
    2. For non-terminal states, it calculates a weighted linear function using features of the state. 
    The features we look at are 2 in a row/col/diagonal where the 3rd square is empty. We assume that
    the more of these positions we have, the higher the chance of winning.
    We need to be careful that the utility of the heuristic stays between [-1,1]. 
    Across legal 3×3 positions, the absolute difference between the two players’ open two-in-a-row counts is at most 3. I weigh the count by 0.3, 
    guaranteeing that is in the needed range.
    
    Function Returns: heuristic value, terminal?"""
    
    # terminal state?
    u = utility(state, player)
    if u is not None: return u, True
      
    score = 0
    board = np.array(state).reshape((3,3))
    diagonals = np.array([[board[i][i] for i in range(len(board))], 
                          [board[i][len(board)-i-1] for i in range(len(board))]])
    
    for a_board in [board, np.transpose(board), diagonals]:
        for row in a_board:
            if sum(row == player) == 2 and any(row ==' '): score += .3
            if sum(row == other(player)) == 2 and any(row ==' '): score -= .3
    
    return score, False

In [20]:
from IPython.display import Markdown, display

display(Markdown(r"""
### Giải thích `eval_fun(state, player)`

- Với trạng thái kết thúc, hàm trả về utility: `+1` nếu `player` thắng, `-1` nếu thua và `0` nếu hòa. Cờ `terminal=True` cho biết không cần mở rộng nút.
- Với trạng thái chưa kết thúc, hàm xét 8 đường thắng (3 hàng, 3 cột và 2 đường chéo).
- Mỗi đường có đúng 2 quân của `player` và 1 ô trống được cộng `+0.3`; đường có 2 quân đối thủ và 1 ô trống bị trừ `0.3`.
- Trọng số `0.3` được chọn để giữ heuristic trong `[-1, 1]` trên các trạng thái hợp lệ của Tic-Tac-Toe 3×3. Một trạng thái có thể đồng thời có nhiều đường hai quân, vì vậy trọng số `0.4` trước đó có thể tạo giá trị `±1.2` và vi phạm giới hạn đã nêu.

### Ý nghĩa `cutoff`

`cutoff=None` tìm đến trạng thái kết thúc; `cutoff=2` và `cutoff=4` dừng mở rộng khi đạt độ sâu tương ứng rồi dùng `eval_fun` ước lượng. Cutoff nhỏ tiết kiệm nút và thời gian nhưng có thể bỏ sót phản hồi xa hơn; cutoff lớn tốn tài nguyên hơn nhưng nhìn trước được nhiều nước đi hơn.
"""))



### Giải thích `eval_fun(state, player)`

- Với trạng thái kết thúc, hàm trả về utility: `+1` nếu `player` thắng, `-1` nếu thua và `0` nếu hòa. Cờ `terminal=True` cho biết không cần mở rộng nút.
- Với trạng thái chưa kết thúc, hàm xét 8 đường thắng (3 hàng, 3 cột và 2 đường chéo).
- Mỗi đường có đúng 2 quân của `player` và 1 ô trống được cộng `+0.3`; đường có 2 quân đối thủ và 1 ô trống bị trừ `0.3`.
- Trọng số `0.3` được chọn để giữ heuristic trong `[-1, 1]` trên các trạng thái hợp lệ của Tic-Tac-Toe 3×3. Một trạng thái có thể đồng thời có nhiều đường hai quân, vì vậy trọng số `0.4` trước đó có thể tạo giá trị `±1.2` và vi phạm giới hạn đã nêu.

### Ý nghĩa `cutoff`

`cutoff=None` tìm đến trạng thái kết thúc; `cutoff=2` và `cutoff=4` dừng mở rộng khi đạt độ sâu tương ứng rồi dùng `eval_fun` ước lượng. Cutoff nhỏ tiết kiệm nút và thời gian nhưng có thể bỏ sót phản hồi xa hơn; cutoff lớn tốn tài nguyên hơn nhưng nhìn trước được nhiều nước đi hơn.


In [21]:
board = empty_board() 
show_board(board)
print(f"eval for x: {eval_fun(board)}")
print(f"eval for o: {eval_fun(board, 'o')}")

board = empty_board() 
board[0] = 'x'
board[1] = 'x'
board[2] = 'x' 
show_board(board)
print(f"eval for x: {eval_fun(board)}")
print(f"eval for o: {eval_fun(board, 'o')}")

board = empty_board() 
board[0] = 'x'
board[1] = 'x'
board[3] = 'x' 
board[4] = 'o'
board[8] = 'o'
show_board(board)
print(f"eval for x: {eval_fun(board)}")
print(f"eval for o: {eval_fun(board, 'o')}")

board = empty_board() 
board[0] = 'x'
board[1] = 'o'
board[3] = 'x' 
board[4] = 'o'
show_board(board)
print(f"eval for x: {eval_fun(board)}")
print(f"eval for o: {eval_fun(board, 'o')}")

[[' ' ' ' ' ']
 [' ' ' ' ' ']
 [' ' ' ' ' ']]
eval for x: (0, False)
eval for o: (0, False)
[['x' 'x' 'x']
 [' ' ' ' ' ']
 [' ' ' ' ' ']]
eval for x: (1, True)
eval for o: (-1, True)
[['x' 'x' ' ']
 ['x' 'o' ' ']
 [' ' ' ' 'o']]
eval for x: (0.6, False)
eval for o: (-0.6, False)
[['x' 'o' ' ']
 ['x' 'o' ' ']
 [' ' ' ' ' ']]
eval for x: (0.0, False)
eval for o: (0.0, False)


## Search with Cutoff

We add a cutoff to the Recursive DFS algorithm for Minimax Search with Alpha-Beta Pruning (see AIMA page 156ff). We use the heuristic evaluation function and the back-up the value using minimax search with alpha-beta pruning to determine the next move.

In [22]:
import math

# global variables
DEBUG = 1 # 1 ... count nodes, 2 ... debug each node
COUNT = 0

def alpha_beta_search(board, cutoff = None, player = 'x'):
    """start the search. cutoff = None is minimax search with alpha-beta pruning."""
    global DEBUG, COUNT
    COUNT = 0

    value, move = max_value_ab(board, player, -math.inf, +math.inf, 0, cutoff)
    
    if DEBUG >= 1: print(f"Number of nodes searched (cutoff = {cutoff}): {COUNT}") 
    
    return {"move": move, "value": value}

def max_value_ab(state, player, alpha, beta, depth, cutoff):
    """player's best move."""
    global DEBUG, COUNT
    COUNT += 1
    
    # cut off and terminal test
    v, terminal = eval_fun(state, player)
    if((cutoff is not None and depth >= cutoff) or terminal): 
        if(terminal): 
            alpha, beta = v, v
        if DEBUG >= 2: print(f"stopped at {depth}: {state} term: {terminal} eval: {v} [{alpha}, {beta}]" ) 
        return v, None
    
    v, move = -math.inf, None

    # check all possible actions in the state, update alpha and return move with the largest value
    for a in actions(state):
        v2, a2 = min_value_ab(result(state, player, a), player, alpha, beta, depth + 1, cutoff)
        if v2 > v:
            v, move = v2, a
            alpha = max(alpha, v)
        if v >= beta: return v, move
    
    return v, move

def min_value_ab(state, player, alpha, beta, depth, cutoff):
    """opponent's best response."""
    global DEBUG, COUNT
    COUNT += 1
    
    # cut off and terminal test
    v, terminal = eval_fun(state, player)
    if((cutoff is not None and depth >= cutoff) or terminal): 
        if(terminal): 
            alpha, beta = v, v
        if DEBUG >= 2: print(f"stopped at {depth}: {state} term: {terminal} eval: {v} [{alpha}, {beta}]" ) 
        return v, None
    
    v, move = +math.inf, None

    # check all possible actions in the state, update beta and return move with the smallest value
    for a in actions(state):
        v2, a2 = max_value_ab(result(state, other(player), a), player, alpha, beta, depth + 1, cutoff)
        if v2 < v:
            v, move = v2, a
            beta = min(beta, v)
        if v <= alpha: return v, move
    
    return v, move

## Some Tests

### x is about to win (play 8)

In [23]:
board = empty_board() 
board[0] = 'x'
board[1] = 'o'
board[3] = 'o'
board[4] = 'x'

print("Board:")
show_board(board)

print()
%time display(alpha_beta_search(board, 2))

print()
%time display(alpha_beta_search(board, 4))

print()
%time display(alpha_beta_search(board))

Board:
[['x' 'o' ' ']
 ['o' 'x' ' ']
 [' ' ' ' ' ']]

Number of nodes searched (cutoff = 2): 13


{'move': 8, 'value': 1}

CPU times: total: 15.6 ms
Wall time: 2 ms

Number of nodes searched (cutoff = 4): 47


{'move': 2, 'value': 1}

CPU times: total: 0 ns
Wall time: 4.08 ms

Number of nodes searched (cutoff = None): 61


{'move': 2, 'value': 1}

CPU times: total: 0 ns
Wall time: 3.7 ms


### o is about to win

In [24]:
board = empty_board() 
board[0] = 'o'
board[1] = 'o'
board[3] = 'o'
board[4] = 'x'
board[8] = 'x'

print("Board:")
show_board(board)

print()
%time display(alpha_beta_search(board, 2))
print()
%time display(alpha_beta_search(board))

Board:
[['o' 'o' ' ']
 ['o' 'x' ' ']
 [' ' ' ' 'x']]

Number of nodes searched (cutoff = 2): 11


{'move': 2, 'value': -1}

CPU times: total: 0 ns
Wall time: 1.62 ms

Number of nodes searched (cutoff = None): 15


{'move': 2, 'value': -1}

CPU times: total: 0 ns
Wall time: 1.83 ms


### x can draw if it chooses 7

In [25]:
board = empty_board() 
board[0] = 'x'
board[1] = 'o'
board[2] = 'x'
board[4] = 'o'

print("Board:")
show_board(board)

print()
%time display(alpha_beta_search(board, 2))
print()
%time display(alpha_beta_search(board, 4))
print()
%time display(alpha_beta_search(board))

Board:
[['x' 'o' 'x']
 [' ' 'o' ' ']
 [' ' ' ' ' ']]

Number of nodes searched (cutoff = 2): 21


{'move': 7, 'value': -0.3}

CPU times: total: 15.6 ms
Wall time: 2.44 ms

Number of nodes searched (cutoff = 4): 81


{'move': 7, 'value': 0}

CPU times: total: 0 ns
Wall time: 5.1 ms

Number of nodes searched (cutoff = None): 101


{'move': 7, 'value': 0}

CPU times: total: 15.6 ms
Wall time: 9.97 ms


### Empty board: Only a draw an be guaranteed

In [26]:
board = empty_board() 

print("Board:")
show_board(board)


print()
%time display(alpha_beta_search(board, 2))
print()
%time display(alpha_beta_search(board, 4))
print()
%time display(alpha_beta_search(board))

Board:
[[' ' ' ' ' ']
 [' ' ' ' ' ']
 [' ' ' ' ' ']]

Number of nodes searched (cutoff = 2): 26


{'move': 0, 'value': 0}

CPU times: total: 15.6 ms
Wall time: 3.68 ms

Number of nodes searched (cutoff = 4): 541


{'move': 4, 'value': 0.0}

CPU times: total: 31.2 ms
Wall time: 32.8 ms

Number of nodes searched (cutoff = None): 18297


{'move': 0, 'value': 0}

CPU times: total: 609 ms
Wall time: 605 ms


### A bad situation

In [27]:
board = empty_board() 
board[0] = 'o'
board[2] = 'x'
board[8] = 'o'

print("Board:")
show_board(board)

print()
%time display(alpha_beta_search(board, 2))
print()
%time display(alpha_beta_search(board, 4))
print()
%time display(alpha_beta_search(board))

Board:
[['o' ' ' 'x']
 [' ' ' ' ' ']
 [' ' ' ' 'o']]

Number of nodes searched (cutoff = 2): 26


{'move': 4, 'value': -0.6}

CPU times: total: 0 ns
Wall time: 2.53 ms

Number of nodes searched (cutoff = 4): 148


{'move': 1, 'value': -1}

CPU times: total: 15.6 ms
Wall time: 8.06 ms

Number of nodes searched (cutoff = None): 238


{'move': 1, 'value': -1}

CPU times: total: 0 ns
Wall time: 9.5 ms


## Experiments



### Minimax with Alpha-Beta Pruning vs. Random

In [28]:
def heuristic2_player(board, player = 'x'):
    return alpha_beta_search(board, cutoff = 2, player = player)["move"]

def heuristic4_player(board, player = 'x'):
    return alpha_beta_search(board, cutoff = 4, player = player)["move"]

def alpha_beta_player(board, player = 'x'):
    return alpha_beta_search(board, cutoff = None, player = player)["move"]

DEBUG = 1
print("heuristic2 vs. random:")
display(play(heuristic2_player, random_player, N = 3))

heuristic2 vs. random:
Number of nodes searched (cutoff = 2): 26
Number of nodes searched (cutoff = 2): 37
Number of nodes searched (cutoff = 2): 13
Number of nodes searched (cutoff = 2): 25
Number of nodes searched (cutoff = 2): 21
Number of nodes searched (cutoff = 2): 13
Number of nodes searched (cutoff = 2): 4
Number of nodes searched (cutoff = 2): 26
Number of nodes searched (cutoff = 2): 24
Number of nodes searched (cutoff = 2): 10


{'x': 2, 'o': 1, 'd': 0}

In [29]:

import pandas as pd, time, random
from collections import Counter

def run_tournament(agent, n=100, agent_first=True, seed=2026):
    counts=Counter()
    t0=time.perf_counter()
    for i in range(n):
        random.seed(seed+i)
        board=empty_board()
        current='x'
        # Agent plays as X when agent_first=True, otherwise as O.
        while not terminal(board):
            if (current=='x') == agent_first:
                move=agent(board,current)
            else:
                move=random_player(board,current)
            board=result(board,current,move)
            current=other(current)
        counts[check_board(board)]+=1
    return {"Agent":agent.__name__,"Vị trí":"X (đi trước)" if agent_first else "O (đi sau)",
            "Agent thắng":counts['x'] if agent_first else counts['o'],
            "Random thắng":counts['o'] if agent_first else counts['x'],
            "Hòa":counts['d'],"Số ván":n,"Thời gian (s)":round(time.perf_counter()-t0,3)}

DEBUG=0
tournament_rows=[]
for agent in [heuristic2_player,heuristic4_player]:
    tournament_rows.append(run_tournament(agent,100,True,1000))
    tournament_rows.append(run_tournament(agent,100,False,2000))
match_df=pd.DataFrame(tournament_rows)
display(match_df)


,Agent,Vị trí,Agent thắng,Random thắng,Hòa,Số ván,Thời gian (s)
0,heuristic2_player,X (đi trước),94,0,6,100,0.321
1,heuristic2_player,O (đi sau),83,1,16,100,0.295
2,heuristic4_player,X (đi trước),98,0,2,100,4.023
3,heuristic4_player,O (đi sau),86,0,14,100,2.306


### Heuristic vs. Minimax with Alpha-Beta Pruning

In [30]:
# So sánh Full Alpha-Beta, cutoff=2 và cutoff=4 trên cùng 5 trạng thái.
# COUNT là số nút được đếm bên trong alpha_beta_search.
import time
import pandas as pd

test_boards = {
    "X sắp thắng": ['x','o',' ','o','x',' ',' ',' ',' '],
    "O sắp thắng": ['o','o',' ','o','x',' ',' ',' ','x'],
    "X có thể hòa": ['x','o','x',' ','o',' ',' ',' ',' '],
    "Bàn trống": empty_board(),
    "Tình huống bất lợi": ['o',' ','x',' ',' ',' ',' ',' ','o']
}

configurations = [
    ("Alpha-Beta đầy đủ", None),
    ("Heuristic Alpha-Beta", 2),
    ("Heuristic Alpha-Beta", 4)
]

comparison_rows = []
for state_name, board in test_boards.items():
    for method, cutoff in configurations:
        DEBUG = 0
        start = time.perf_counter()
        output = alpha_beta_search(board, cutoff=cutoff, player='x')
        elapsed_ms = (time.perf_counter() - start) * 1000
        comparison_rows.append({
            "Trạng thái": state_name,
            "Phương pháp": method,
            "Độ sâu/cấu hình": "None (đến terminal)" if cutoff is None else cutoff,
            "Nước đi": output["move"],
            "Giá trị": round(output["value"], 4),
            "Số nút": COUNT,
            "Thời gian (ms)": round(elapsed_ms, 4),
            "Nhận xét": (
                "Tìm đến terminal, kết quả chính xác"
                if cutoff is None else
                "Nhanh hơn nhưng tầm nhìn ngắn"
                if cutoff == 2 else
                "Tầm nhìn sâu hơn, chi phí cao hơn"
            )
        })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)


,Trạng thái,Phương pháp,Độ sâu/cấu hình,Nước đi,Giá trị,Số nút,Thời gian (ms),Nhận xét
0,X sắp thắng,Alpha-Beta đầy đủ,None (đến terminal),2,1.0,61,2.2797,"Tìm đến terminal, kết quả chính xác"
1,X sắp thắng,Heuristic Alpha-Beta,2,8,1.0,13,0.7137,Nhanh hơn nhưng tầm nhìn ngắn
2,X sắp thắng,Heuristic Alpha-Beta,4,2,1.0,47,2.1409,"Tầm nhìn sâu hơn, chi phí cao hơn"
3,O sắp thắng,Alpha-Beta đầy đủ,None (đến terminal),2,-1.0,15,0.4392,"Tìm đến terminal, kết quả chính xác"
4,O sắp thắng,Heuristic Alpha-Beta,2,2,-1.0,11,0.3718,Nhanh hơn nhưng tầm nhìn ngắn
5,O sắp thắng,Heuristic Alpha-Beta,4,2,-1.0,15,0.4451,"Tầm nhìn sâu hơn, chi phí cao hơn"
6,X có thể hòa,Alpha-Beta đầy đủ,None (đến terminal),7,0.0,101,3.4079,"Tìm đến terminal, kết quả chính xác"
7,X có thể hòa,Heuristic Alpha-Beta,2,7,-0.3,21,0.9336,Nhanh hơn nhưng tầm nhìn ngắn
8,X có thể hòa,Heuristic Alpha-Beta,4,7,0.0,81,3.3124,"Tầm nhìn sâu hơn, chi phí cao hơn"
9,Bàn trống,Alpha-Beta đầy đủ,None (đến terminal),0,0.0,18297,546.4929,"Tìm đến terminal, kết quả chính xác"


### Heuristic vs. Heuristic

In [31]:

# Đối đầu giữa hai mức cutoff; đổi vị trí X/O để giảm thiên lệch đi trước.
def head_to_head(agent_x,agent_o,n=100,seed=2026):
    counts=Counter()
    t0=time.perf_counter()
    for i in range(n):
        random.seed(seed+i)
        board=empty_board(); current='x'
        while not terminal(board):
            agent=agent_x if current=='x' else agent_o
            move=agent(board,current)
            board=result(board,current,move)
            current=other(current)
        counts[check_board(board)]+=1
    return {"X wins":counts['x'],"O wins":counts['o'],"Draws":counts['d'],
            "Games":n,"Elapsed (s)":round(time.perf_counter()-t0,3)}

head_to_head_df=pd.DataFrame([
    {"Match":"Cutoff 2 (X) vs Cutoff 4 (O)",**head_to_head(heuristic2_player,heuristic4_player)},
    {"Match":"Cutoff 4 (X) vs Cutoff 2 (O)",**head_to_head(heuristic4_player,heuristic2_player)}
])
display(head_to_head_df)


,Match,X wins,O wins,Draws,Games,Elapsed (s)
0,Cutoff 2 (X) vs Cutoff 4 (O),0,0,100,100,1.981
1,Cutoff 4 (X) vs Cutoff 2 (O),0,0,100,100,4.053


__Idea:__ Start experiments with different boards that already have a few x's and o's randomly placed on them.

## Kết luận thực nghiệm bổ sung

- `cutoff=2` giảm chi phí tính toán nhưng tầm nhìn ngắn nên dễ bỏ sót phản hồi chiến thuật của đối thủ.
- `cutoff=4` xét sâu hơn, thường chọn nước đi có chất lượng chiến thuật tốt hơn nhưng duyệt nhiều nút và tốn thời gian hơn.
- `cutoff=None` tìm đến trạng thái kết thúc nên cho giá trị minimax chính xác trên Tic-Tac-Toe 3×3; chi phí tăng mạnh hơn so với tìm kiếm giới hạn.
- Bảng `match_df` thống kê 100 ván cho mỗi vị trí; `comparison_df` so sánh nước đi/thời gian trên cùng bộ trạng thái; `head_to_head_df` so sánh trực tiếp hai cutoff.
- Lưu ý: hai heuristic agent trong đối đầu là tất định; lặp lại cùng trạng thái đầu và cùng thứ tự duyệt sẽ tạo lại cùng ván. Vì vậy số lần lặp ở `head_to_head_df` không phải các mẫu ngẫu nhiên độc lập; cần đọc kết quả theo từng vị trí X/O.


In [32]:
# Kiểm tra tính hợp lệ của các nước đi
assert set(comparison_df["Độ sâu/cấu hình"]) == {
    "None (đến terminal)", 2, 4
}
assert comparison_df["Nước đi"].notna().all()
assert (comparison_df["Số nút"] > 0).all()

# Heuristic values for the tested legal positions must stay within the utility scale.
for _board in test_boards.values():
    _value, _is_terminal = eval_fun(_board, "x")
    assert -1.0 <= _value <= 1.0

# Tạo thư mục result
import os

RESULT_DIR = "result"
os.makedirs(RESULT_DIR, exist_ok=True)

# 1. Lưu bảng so sánh Alpha-Beta
comparison_df.to_csv(
    os.path.join(RESULT_DIR, "bai4_alpha_beta_comparison.csv"),
    index=False,
    encoding="utf-8-sig"
)

# 2. Lưu kết quả Heuristic vs Random
match_df.to_csv(
    os.path.join(RESULT_DIR, "bai4_heuristic_random_tournament.csv"),
    index=False,
    encoding="utf-8-sig"
)

# 3. Lưu kết quả đối đầu giữa các Heuristic Agent
head_to_head_df.to_csv(
    os.path.join(RESULT_DIR, "bai4_heuristic_head_to_head.csv"),
    index=False,
    encoding="utf-8-sig"
)

print("Bài 4: Kiểm tra thành công, đã lưu 3 bảng vào thư mục result.")
# Exhaustively enumerate reachable positions to verify the heuristic range.
from collections import deque
_reachable = {tuple(empty_board())}
_frontier = deque([tuple(empty_board())])
while _frontier:
    _state = _frontier.popleft()
    if terminal(list(_state)):
        continue
    _player = "x" if _state.count("x") == _state.count("o") else "o"
    for _action in actions(list(_state)):
        _child = tuple(result(list(_state), _player, _action))
        if _child not in _reachable:
            _reachable.add(_child)
            _frontier.append(_child)
assert all(-1.0 <= eval_fun(list(_state), "x")[0] <= 1.0 for _state in _reachable)
print(f"Heuristic range check passed on {len(_reachable):,} reachable states.")


Bài 4: Kiểm tra thành công, đã lưu 3 bảng vào thư mục result.
Heuristic range check passed on 5,478 reachable states.


In [33]:
# ĐÁNH GIÁ CHUẨN HÓA: mỗi agent đấu Random 200 ván (100 X + 100 O).
# Thời gian chỉ tính phần agent được đánh giá suy nghĩ.
import time
import random
import os
import pandas as pd
from collections import Counter

def run_standard_tournament(agent, agent_name=None, games_per_side=100, seed=20261112):
    name = agent_name or getattr(agent, "__name__", "Agent")
    rows = []
    total_agent_time = 0.0
    total_agent_moves = 0
    total_wins = total_draws = total_losses = 0

    for agent_first in (True, False):
        counts = Counter()
        side_time = 0.0
        side_moves = 0

        for game_idx in range(games_per_side):
            random.seed(seed + (0 if agent_first else 100000) + game_idx)
            board = empty_board()
            current = "x"

            while not terminal(board):
                is_agent_turn = ((current == "x") == agent_first)
                if is_agent_turn:
                    t0 = time.perf_counter()
                    move = agent(board, current)
                    side_time += time.perf_counter() - t0
                    side_moves += 1
                else:
                    move = random_player(board, current)

                board = result(board, current, move)
                current = other(current)

            outcome = check_board(board)
            agent_symbol = "x" if agent_first else "o"
            random_symbol = "o" if agent_first else "x"
            if outcome == "d":
                counts["draw"] += 1
            elif outcome == agent_symbol:
                counts["win"] += 1
            else:
                counts["loss"] += 1

        total_agent_time += side_time
        total_agent_moves += side_moves
        total_wins += counts["win"]
        total_draws += counts["draw"]
        total_losses += counts["loss"]
        rows.append({
            "Agent": name,
            "Vị trí": "X (đi trước)" if agent_first else "O (đi sau)",
            "Thắng": counts["win"], "Hòa": counts["draw"], "Thua": counts["loss"],
            "Số ván": games_per_side,
            "Tổng thời gian agent (ms)": round(side_time * 1000, 4),
            "Số nước agent": side_moves,
            "Thời gian TB/nước (ms)": round(side_time * 1000 / side_moves, 6) if side_moves else 0
        })

    summary = {
        "Agent": name, "Thắng": total_wins, "Hòa": total_draws, "Thua": total_losses,
        "Tổng số ván": 2 * games_per_side,
        "Tổng thời gian agent (ms)": round(total_agent_time * 1000, 4),
        "Tổng số nước agent": total_agent_moves,
        "Thời gian TB/nước (ms)": round(total_agent_time * 1000 / total_agent_moves, 6) if total_agent_moves else 0
    }
    return pd.DataFrame(rows), summary

standard_agents = [
    ("Alpha-Beta đầy đủ", alpha_beta_player),
    ("Heuristic cutoff=2", heuristic2_player),
    ("Heuristic cutoff=4", heuristic4_player),
]
standard_detail_frames = []
standard_summary_rows = []
for idx, (agent_name, agent_fn) in enumerate(standard_agents):
    detail, summary = run_standard_tournament(agent_fn, agent_name, seed=20261112 + idx * 1000)
    standard_detail_frames.append(detail)
    standard_summary_rows.append(summary)

standard_tournament_detail_df = pd.concat(standard_detail_frames, ignore_index=True)
standard_tournament_summary_df = pd.DataFrame(standard_summary_rows)
display(standard_tournament_detail_df)
display(standard_tournament_summary_df)

os.makedirs("result", exist_ok=True)
standard_tournament_detail_df.to_csv("result/bai4_standard_tournament_detail.csv", index=False, encoding="utf-8-sig")
standard_tournament_summary_df.to_csv("result/bai4_standard_tournament_summary.csv", index=False, encoding="utf-8-sig")


,Agent,Vị trí,Thắng,Hòa,Thua,Số ván,Tổng thời gian agent (ms),Số nước agent,Thời gian TB/nước (ms)
0,Alpha-Beta đầy đủ,X (đi trước),99,1,0,100,57491.8194,339,169.592388
1,Alpha-Beta đầy đủ,O (đi sau),80,20,0,100,11065.1078,361,30.651268
2,Heuristic cutoff=2,X (đi trước),93,7,0,100,310.3885,325,0.955042
3,Heuristic cutoff=2,O (đi sau),81,16,3,100,277.2504,329,0.842706
4,Heuristic cutoff=4,X (đi trước),100,0,0,100,3929.4834,401,9.799210
5,Heuristic cutoff=4,O (đi sau),74,26,0,100,2415.3142,353,6.842250


,Agent,Thắng,Hòa,Thua,Tổng số ván,Tổng thời gian agent (ms),Tổng số nước agent,Thời gian TB/nước (ms)
0,Alpha-Beta đầy đủ,179,21,0,200,68556.9272,700,97.938467
1,Heuristic cutoff=2,174,23,3,200,587.6389,654,0.898530
2,Heuristic cutoff=4,174,26,0,200,6344.7976,754,8.414851
